# SmartPDS — Layer 3: Fraud Detection
**Team Kahunas | DevsHouse 26 | Open Innovation**

Detects fraudulent PDS transactions using Isolation Forest — an unsupervised anomaly detection algorithm that requires zero labeled fraud examples. Fraud signals are engineered from the gap between LP-allocated quantities and actual collected quantities, combined with transaction behaviour patterns from the raw data.

**Input:** `transactions.csv`, `ration_cards.csv`, `fps_data.csv`, `villages.csv`, `allocation_plan.csv`

**Output:** `fraud_alerts.csv` (input for Layer 4 — GIS Dashboard)

**Pipeline:** Transaction Signals + Allocation Gap -> Feature Engineering -> Isolation Forest -> SHAP -> Fraud Alerts Export

## 0. Dependencies

In [ ]:
!pip install scikit-learn shap pandas numpy matplotlib seaborn -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import shap

warnings.filterwarnings('ignore')
np.random.seed(42)

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#f5f5f5',
    'axes.grid':        True,
    'grid.alpha':       0.4,
    'font.size':        11,
    'axes.spines.top':   False,
    'axes.spines.right': False,
})

## 1. Load All Source Files

All five files must be in the same working directory. `allocation_plan.csv` is the Layer 2 output.

In [ ]:
txn        = pd.read_csv('transactions.csv')
cards      = pd.read_csv('ration_cards.csv')
fps        = pd.read_csv('fps_data.csv')
villages   = pd.read_csv('villages.csv')
allocation = pd.read_csv('allocation_plan.csv')

print(f'Transactions  : {len(txn):>6} rows | columns: {list(txn.columns)}')
print(f'Ration cards  : {len(cards):>6} rows | columns: {list(cards.columns)}')
print(f'FPS data      : {len(fps):>6} rows | columns: {list(fps.columns)}')
print(f'Villages      : {len(villages):>6} rows | columns: {list(villages.columns)}')
print(f'Allocation    : {len(allocation):>6} rows | columns: {list(allocation.columns)}')

## 2. Village-Level Fraud Signal Engineering

Seven fraud signals are computed per village. Each captures a distinct fraud mechanism — dealer diversion, ghost beneficiaries, offline evasion, bulk collection, and price manipulation.

In [ ]:
rice_txn = txn[txn['commodity'] == 'rice'].copy()

# Base transaction aggregation per village
txn_agg = (
    rice_txn.groupby('village_id')
    .agg(
        total_collected_kg  = ('quantity',             'sum'),
        total_txn_count     = ('txn_id',               'count'),
        total_amount        = ('amount',               'sum'),
        avg_delay           = ('delivery_delay_days',  'mean'),
        max_delay           = ('delivery_delay_days',  'max'),
        offline_pct         = ('offline_flag',         'mean'),
        aadhaar_pct         = ('auth_mode', lambda x: (x == 'aadhaar').mean()),
    ).reset_index()
)

# Card-level signals per village
card_agg = (
    cards.groupby('village_id')
    .agg(
        registered_cards = ('card_id',        'count'),
        bpl_count        = ('bpl_status',      'sum'),
        aadhaar_coverage = ('has_aadhaar',     'mean'),
        avg_hh_size      = ('household_size',  'mean'),
    ).reset_index()
)

# Merge everything
fraud_df = txn_agg.merge(card_agg,  on='village_id', how='left')
fraud_df = fraud_df.merge(allocation[['village_id','allocated_kg','forecasted_demand_kg','shortage_risk']],
                          on='village_id', how='left')
fraud_df = fraud_df.merge(villages[['village_id','village_name','district']], on='village_id', how='left')

print(f'Base fraud feature frame: {fraud_df.shape}')
fraud_df.head()

In [ ]:
# Signal 1: collection_ratio
# Actual collected / LP allocated. Below 0.60 = strong diversion signal.
fraud_df['collection_ratio'] = (
    fraud_df['total_collected_kg'] /
    fraud_df['allocated_kg'].replace(0, np.nan)
).clip(0, 2).fillna(1.0)

# Signal 2: ghost_ratio
# Transactions per registered card. > 1.0 means more transactions than cards = ghost beneficiaries.
fraud_df['ghost_ratio'] = (
    fraud_df['total_txn_count'] /
    fraud_df['registered_cards'].replace(0, np.nan)
).fillna(1.0)

# Signal 3: amount_per_kg
# Actual price charged per kg. Deviation from official issue price (~Rs 2/kg for rice) = overcharging.
fraud_df['amount_per_kg'] = (
    fraud_df['total_amount'] /
    fraud_df['total_collected_kg'].replace(0, np.nan)
).fillna(0)

# Signal 4: offline_pct (already computed — high offline = hard to audit)

# Signal 5: delay_score
# Normalised delivery delay. Very long delays often indicate stock is being held for diversion.
fraud_df['delay_score'] = (
    fraud_df['avg_delay'] / fraud_df['avg_delay'].max()
).fillna(0)

# Signal 6: aadhaar_gap
# Difference between card-level Aadhaar coverage and transaction-level Aadhaar auth rate.
# Large gap means registered cards have Aadhaar but transactions bypass it.
fraud_df['aadhaar_gap'] = (
    fraud_df['aadhaar_coverage'] - fraud_df['aadhaar_pct']
).clip(0, 1).fillna(0)

# Signal 7: demand_collection_gap
# (Forecasted demand - collected) / forecasted demand.
# High gap when allocation was sufficient = stock went elsewhere.
fraud_df['demand_collection_gap'] = (
    (fraud_df['forecasted_demand_kg'] - fraud_df['total_collected_kg']) /
    fraud_df['forecasted_demand_kg'].replace(0, np.nan)
).clip(-0.5, 1).fillna(0)

FRAUD_FEATURES = [
    'collection_ratio',
    'ghost_ratio',
    'amount_per_kg',
    'offline_pct',
    'delay_score',
    'aadhaar_gap',
    'demand_collection_gap',
]

print(f'Fraud signals engineered: {len(FRAUD_FEATURES)}')
print(f'Null check: {fraud_df[FRAUD_FEATURES].isnull().sum().sum()} nulls')
print()
fraud_df[FRAUD_FEATURES].describe().round(4)

## 3. Exploratory Analysis of Fraud Signals

Distribution of each signal across villages. Outliers in these distributions are the fraud candidates.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
fig.suptitle('SmartPDS — Fraud Signal Distributions', fontsize=14, fontweight='bold')
axes = axes.flatten()

signal_labels = {
    'collection_ratio':       'Collection Ratio (collected/allocated)',
    'ghost_ratio':            'Ghost Ratio (txns/registered cards)',
    'amount_per_kg':          'Amount per kg (Rs)',
    'offline_pct':            'Offline Transaction %',
    'delay_score':            'Delivery Delay Score',
    'aadhaar_gap':            'Aadhaar Coverage Gap',
    'demand_collection_gap':  'Demand-Collection Gap',
}

for i, (feat, label) in enumerate(signal_labels.items()):
    axes[i].hist(fraud_df[feat], bins=20, color='#2c7bb6', edgecolor='white', alpha=0.85)
    axes[i].set_title(label, fontsize=10)
    axes[i].set_ylabel('Villages')
    q95 = fraud_df[feat].quantile(0.95)
    q05 = fraud_df[feat].quantile(0.05)
    axes[i].axvline(q95, color='#d7191c', linestyle='--', lw=1.2, label='95th pct')
    axes[i].axvline(q05, color='#fdae61', linestyle='--', lw=1.2, label='5th pct')
    axes[i].legend(fontsize=8)

axes[-1].set_visible(False)
plt.tight_layout()
plt.savefig('fraud_signals_eda.png', dpi=150, bbox_inches='tight')
plt.show()

print('Correlation matrix of fraud signals:')
print(fraud_df[FRAUD_FEATURES].corr().round(3).to_string())

## 4. Feature Scaling

Isolation Forest is sensitive to feature scale when contamination rates differ across features. StandardScaler normalises all signals to zero mean and unit variance before training.

In [ ]:
X = fraud_df[FRAUD_FEATURES].values

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'Feature matrix shape : {X_scaled.shape}')
print(f'Mean after scaling   : {X_scaled.mean(axis=0).round(6)}')
print(f'Std after scaling    : {X_scaled.std(axis=0).round(4)}')

## 5. Isolation Forest Training

`contamination=0.05` matches the known ~5% fraud rate injected during data generation. `n_estimators=200` gives stable anomaly scores. Each tree isolates points by random feature splits — anomalies are isolated in fewer splits and receive lower (more negative) anomaly scores.

In [ ]:
iso_forest = IsolationForest(
    n_estimators  = 200,
    contamination = 0.05,
    max_samples   = 'auto',
    max_features  = 1.0,
    random_state  = 42,
    n_jobs        = -1,
)

iso_forest.fit(X_scaled)

# Predictions: -1 = anomaly (fraud), 1 = normal
predictions   = iso_forest.predict(X_scaled)
anomaly_scores = iso_forest.decision_function(X_scaled)  # lower = more anomalous

fraud_df['anomaly_score']  = np.round(anomaly_scores, 6)
fraud_df['is_fraud_flag']  = (predictions == -1).astype(int)
fraud_df['fraud_label']    = fraud_df['is_fraud_flag'].map({1: 'Fraud', 0: 'Normal'})

n_flagged = fraud_df['is_fraud_flag'].sum()
n_total   = len(fraud_df)

print(f'Total villages analysed : {n_total}')
print(f'Flagged as fraud        : {n_flagged} ({n_flagged/n_total*100:.1f}%)')
print(f'Normal                  : {n_total - n_flagged}')
print()
print('Anomaly score statistics:')
print(f'  Min  (most anomalous) : {anomaly_scores.min():.4f}')
print(f'  Max  (most normal)    : {anomaly_scores.max():.4f}')
print(f'  Mean                  : {anomaly_scores.mean():.4f}')
print(f'  Threshold (approx)    : {np.percentile(anomaly_scores, 5):.4f}')

## 6. Performance Evaluation

Since fraud labels were injected during data generation in Layer 1 (`is_fraud` column in transactions.csv), we can compute village-level ground truth and evaluate precision and recall.

In [ ]:
# Build ground truth: a village is fraudulent if any of its transactions were fraud
if 'is_fraud' in txn.columns:
    ground_truth = (
        txn.groupby('village_id')['is_fraud']
        .max()
        .reset_index()
        .rename(columns={'is_fraud': 'true_fraud'})
    )
    fraud_df = fraud_df.merge(ground_truth, on='village_id', how='left')
    fraud_df['true_fraud'] = fraud_df['true_fraud'].fillna(0).astype(int)

    y_true = fraud_df['true_fraud'].values
    y_pred = fraud_df['is_fraud_flag'].values

    print('Classification Report (village level):')
    print(classification_report(y_true, y_pred, target_names=['Normal', 'Fraud']))

    cm = confusion_matrix(y_true, y_pred)
    print('Confusion Matrix:')
    print(f'  True Normal  flagged Normal (TN) : {cm[0,0]}')
    print(f'  True Normal  flagged Fraud  (FP) : {cm[0,1]}')
    print(f'  True Fraud   flagged Normal (FN) : {cm[1,0]}')
    print(f'  True Fraud   flagged Fraud  (TP) : {cm[1,1]}')

    tp = cm[1,1]
    fp = cm[0,1]
    fn = cm[1,0]
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
    fpr       = fp / (cm[0,0] + fp) if (cm[0,0] + fp) > 0 else 0

    print()
    print(f'  Precision         : {precision*100:.1f}%')
    print(f'  Recall            : {recall*100:.1f}%')
    print(f'  False positive rate: {fpr*100:.1f}%')
else:
    print('is_fraud column not found in transactions.csv — skipping ground truth evaluation')
    print('Isolation Forest ran in fully unsupervised mode.')
    fraud_df['true_fraud'] = np.nan

## 7. SHAP Explainability on Anomaly Scores

SHAP TreeExplainer is applied to the Isolation Forest to quantify which fraud signal drove each village's anomaly score. This gives the government auditor a plain-signal explanation for every alert.

In [ ]:
print('Computing SHAP values for Isolation Forest...')

X_shap       = pd.DataFrame(X_scaled, columns=FRAUD_FEATURES)
explainer    = shap.TreeExplainer(iso_forest)
shap_values  = explainer.shap_values(X_shap)

# Top driver feature per village (feature with highest absolute SHAP value)
top_driver_idx   = np.argmax(np.abs(shap_values), axis=1)
top_driver_names = [FRAUD_FEATURES[i] for i in top_driver_idx]
top_driver_shap  = shap_values[np.arange(len(shap_values)), top_driver_idx]

fraud_df['top_driver_feature'] = top_driver_names
fraud_df['top_driver_shap']    = np.round(top_driver_shap, 4)

# Plain-English explanation per flagged village
FEATURE_EXPLANATIONS = {
    'collection_ratio':      'Collected quantity significantly below allocated — possible dealer diversion',
    'ghost_ratio':           'Transactions exceed registered cards — possible ghost beneficiaries',
    'amount_per_kg':         'Price per kg deviates from official issue price — possible overcharging',
    'offline_pct':           'High proportion of offline transactions — authentication bypassed',
    'delay_score':           'Unusually high delivery delays — stock may have been held for diversion',
    'aadhaar_gap':           'Aadhaar coverage present but authentication rate low — bypass suspected',
    'demand_collection_gap': 'Large gap between forecast and collected despite sufficient allocation',
}

fraud_df['alert_reason'] = fraud_df.apply(
    lambda r: FEATURE_EXPLANATIONS.get(r['top_driver_feature'], 'Anomalous pattern detected')
              if r['is_fraud_flag'] == 1 else 'No anomaly',
    axis=1
)

print(f'SHAP values computed for {len(fraud_df)} villages')
print()
print('Top driver feature frequency among flagged villages:')
flagged = fraud_df[fraud_df['is_fraud_flag'] == 1]
print(flagged['top_driver_feature'].value_counts().to_string())

## 8. SHAP Visualizations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('SmartPDS — SHAP Explainability (Isolation Forest)', fontsize=14, fontweight='bold')

# Global SHAP summary
plt.sca(axes[0])
shap.summary_plot(
    shap_values, X_shap,
    feature_names=FRAUD_FEATURES,
    show=False, plot_size=None,
)
axes[0].set_title('Feature impact on anomaly scores — all villages')

# Single flagged village explanation
flagged_indices = fraud_df[fraud_df['is_fraud_flag'] == 1].index.tolist()

if len(flagged_indices) > 0:
    sample_idx  = flagged_indices[0]
    shap_single = shap_values[sample_idx]
    sorted_idx  = np.argsort(np.abs(shap_single))
    feat_names  = [FRAUD_FEATURES[i] for i in sorted_idx]
    feat_shap   = shap_single[sorted_idx]
    bar_colors  = ['#d7191c' if v > 0 else '#2c7bb6' for v in feat_shap]

    axes[1].barh(feat_names, feat_shap, color=bar_colors, edgecolor='white')
    axes[1].axvline(0, color='black', lw=0.8)
    axes[1].set_xlabel('SHAP value (contribution to anomaly score)')

    village_label = fraud_df.loc[sample_idx, 'village_name'] \
        if 'village_name' in fraud_df.columns \
        else fraud_df.loc[sample_idx, 'village_id']

    axes[1].set_title(
        f'Why is {village_label} flagged?\n'
        f'Anomaly score: {fraud_df.loc[sample_idx, "anomaly_score"]:.4f}   '
        f'Top driver: {fraud_df.loc[sample_idx, "top_driver_feature"]}'
    )
else:
    axes[1].text(0.5, 0.5, 'No villages flagged', ha='center', va='center', transform=axes[1].transAxes)

plt.tight_layout()
plt.savefig('fraud_shap.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Fraud Detection Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 13))
fig.suptitle('SmartPDS — Fraud Detection Results', fontsize=15, fontweight='bold')

# Plot 1: Anomaly score distribution with fraud/normal separation
ax = axes[0, 0]
normal  = fraud_df[fraud_df['is_fraud_flag'] == 0]['anomaly_score']
flagged = fraud_df[fraud_df['is_fraud_flag'] == 1]['anomaly_score']
ax.hist(normal,  bins=20, alpha=0.75, color='#1a9641', edgecolor='white', label='Normal')
ax.hist(flagged, bins=20, alpha=0.75, color='#d7191c', edgecolor='white', label='Fraud flagged')
threshold = fraud_df[fraud_df['is_fraud_flag'] == 1]['anomaly_score'].max()
ax.axvline(threshold, color='black', linestyle='--', lw=1.2, label=f'Decision boundary')
ax.set_xlabel('Anomaly score (lower = more anomalous)')
ax.set_ylabel('Number of villages')
ax.set_title('Anomaly Score Distribution')
ax.legend()

# Plot 2: Collection ratio vs ghost ratio scatter — fraud villages highlighted
ax = axes[0, 1]
normal_mask = fraud_df['is_fraud_flag'] == 0
fraud_mask  = fraud_df['is_fraud_flag'] == 1
ax.scatter(fraud_df.loc[normal_mask, 'collection_ratio'],
           fraud_df.loc[normal_mask, 'ghost_ratio'],
           alpha=0.6, s=40, color='#2c7bb6', label='Normal', zorder=2)
ax.scatter(fraud_df.loc[fraud_mask, 'collection_ratio'],
           fraud_df.loc[fraud_mask, 'ghost_ratio'],
           alpha=0.9, s=80, color='#d7191c', marker='X', label='Fraud flagged', zorder=3)
ax.axvline(1.0, color='gray', linestyle=':', lw=1)
ax.axhline(1.0, color='gray', linestyle=':', lw=1)
ax.set_xlabel('Collection ratio (collected / allocated)')
ax.set_ylabel('Ghost ratio (transactions / registered cards)')
ax.set_title('Key Fraud Signals — Collection vs Ghost Ratio')
ax.legend()

# Plot 3: Top driver feature frequency
ax = axes[1, 0]
driver_counts = fraud_df[fraud_df['is_fraud_flag'] == 1]['top_driver_feature'].value_counts()
if len(driver_counts) > 0:
    driver_counts.plot(kind='barh', ax=ax, color='#d7191c', edgecolor='white', alpha=0.85)
    ax.set_xlabel('Number of flagged villages')
    ax.set_title('Primary Fraud Driver Across Flagged Villages')
    ax.invert_yaxis()
else:
    ax.text(0.5, 0.5, 'No fraud flags', ha='center', va='center', transform=ax.transAxes)

# Plot 4: Offline pct vs Aadhaar gap
ax = axes[1, 1]
ax.scatter(fraud_df.loc[normal_mask, 'offline_pct'],
           fraud_df.loc[normal_mask, 'aadhaar_gap'],
           alpha=0.6, s=40, color='#2c7bb6', label='Normal')
ax.scatter(fraud_df.loc[fraud_mask, 'offline_pct'],
           fraud_df.loc[fraud_mask, 'aadhaar_gap'],
           alpha=0.9, s=80, color='#d7191c', marker='X', label='Fraud flagged')
ax.set_xlabel('Offline transaction %')
ax.set_ylabel('Aadhaar coverage gap')
ax.set_title('Authentication Evasion Signals')
ax.legend()

plt.tight_layout()
plt.savefig('fraud_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Flagged Village Alert Table

In [ ]:
alert_cols = [
    'village_id', 'anomaly_score', 'is_fraud_flag',
    'collection_ratio', 'ghost_ratio', 'offline_pct',
    'aadhaar_gap', 'demand_collection_gap',
    'top_driver_feature', 'alert_reason',
]

if 'village_name' in fraud_df.columns:
    alert_cols.insert(1, 'village_name')
if 'district' in fraud_df.columns:
    alert_cols.insert(2, 'district')
if 'true_fraud' in fraud_df.columns and not fraud_df['true_fraud'].isnull().all():
    alert_cols.append('true_fraud')

flagged_alerts = (
    fraud_df[fraud_df['is_fraud_flag'] == 1][alert_cols]
    .sort_values('anomaly_score')
    .reset_index(drop=True)
)

print(f'Fraud alert table ({len(flagged_alerts)} villages):')
print(flagged_alerts.to_string(index=False))

## 11. Export Outputs

`fraud_alerts.csv` is the direct input for Layer 4 — the Streamlit GIS Dashboard. The full village risk table is also exported for the dashboard's map overlay.

In [ ]:
export_cols = [
    'village_id', 'anomaly_score', 'is_fraud_flag', 'fraud_label',
    'collection_ratio', 'ghost_ratio', 'amount_per_kg',
    'offline_pct', 'delay_score', 'aadhaar_gap', 'demand_collection_gap',
    'top_driver_feature', 'top_driver_shap', 'alert_reason',
    'shortage_risk', 'allocated_kg', 'total_collected_kg', 'forecasted_demand_kg',
]

if 'village_name' in fraud_df.columns: export_cols.insert(1, 'village_name')
if 'district'     in fraud_df.columns: export_cols.insert(2, 'district')
if 'true_fraud'   in fraud_df.columns and not fraud_df['true_fraud'].isnull().all():
    export_cols.append('true_fraud')

fraud_df[export_cols].sort_values('anomaly_score').to_csv('fraud_alerts.csv', index=False)

print('Outputs written:')
print('  fraud_alerts.csv   — input for Layer 4 GIS Dashboard')
print()
print('Fraud Detection Summary')
print('-' * 48)
print(f'  Algorithm           : Isolation Forest (sklearn)')
print(f'  Villages analysed   : {len(fraud_df)}')
print(f'  Fraud signals used  : {len(FRAUD_FEATURES)}')
print(f'  Contamination rate  : 5.0%')
print(f'  Villages flagged    : {fraud_df["is_fraud_flag"].sum()}')
print(f'  Flag rate           : {fraud_df["is_fraud_flag"].mean()*100:.1f}%')

if 'true_fraud' in fraud_df.columns and not fraud_df['true_fraud'].isnull().all():
    print(f'  Precision           : {precision*100:.1f}%')
    print(f'  Recall              : {recall*100:.1f}%')
    print(f'  False positive rate : {fpr*100:.1f}%')

print('-' * 48)
print()
print('Primary fraud signals detected:')
if len(driver_counts) > 0:
    for feat, count in driver_counts.items():
        print(f'  {feat:<30} : {count} villages')
print()
print('Next: Layer 4 — Streamlit GIS Dashboard')